[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/091.%20The%20Contract%20Design%20to%20Mitigate%20the%20Bullwhip%20Effect/P91-Tier-2_executed.ipynb)

---


# 91. The Contract Design to Mitigate the Bullwhip Effect
## Tier 2: The Classic Heuristic (Iterative Contract Parameter Optimization)

### Purpose and scope

This notebook uses adaptive iterative local search to choose contract parameters for a four-tier supply chain. The search seeks a low weighted objective that balances a simplified bullwhip-amplification measure against contract implementation cost, subject to parameter and cost limits. The notebook reports the selected parameters, modeled amplification, modeled cost, a convergence trace, and a perturbation-based sensitivity analysis.

The method is a heuristic, not an exact optimization method: it accepts improving local moves and stops after a specified number of consecutive iterations without improvement or when it reaches its iteration limit. It does not guarantee a global optimum. The objective and constraints below define the model actually evaluated by the code.

### Inputs and outputs

- **Inputs:** a four-tier, 12-month example; a seeded random-number generator; baseline amplification; cost coefficients; feasibility limits; and local-search settings.
- **Decision parameters:** 15 values, grouped into three information-sharing coefficients, three cost-sharing ratios, and nine penalty-structure parameters.
- **Outputs:** a feasible parameter vector with its objective value, modeled performance and cost measures, plots, and sensitivity summaries. Numerical outputs are produced when the notebook is run.

### Model assumptions

- Information sharing affects modeled amplification through the mean of its three coefficients. The modeled benefit has diminishing returns.
- Implementation cost is the sum of three specified parameter-based cost components. The model does not include tier-specific operating costs or calculate participants' revenues or profits.
- Cost-sharing ratios and penalty parameters affect implementation cost but do not directly affect modeled amplification. The ratios are interpreted as the upstream share at each adjacent-tier interface; the downstream share is the complement.
- The generated monthly demand series illustrates the example's seasonal, noisy demand context. It is reported and plotted, but it is not an input to the amplification, cost, feasibility, or objective calculations.
- The feasibility checks enforce parameter bounds, a minimum mean information-sharing coefficient, a maximum sum of cost-sharing ratios, and an annual implementation-cost ceiling. They do not establish positive profit for every participant.

### Solution sequence

1. Generate the example's monthly demand and initialize a feasible contract parameter vector.
2. Evaluate the objective, which first checks feasibility and then combines normalized amplification and implementation cost.
3. Perturb a randomly selected subset of the current parameters. The perturbation magnitude decays with iteration and increases after a prolonged lack of improvement.
4. Accept a candidate only when it improves the current objective; retain the best feasible solution found.
5. Stop after the no-improvement limit or the maximum number of iterations, then analyze the result and test nearby parameter vectors.

### What to look for

The convergence plot shows the objective of the current accepted solution, not a proof of global optimality. The parameter and cost plots describe the best solution found by this run. Sensitivity results summarize only the feasible perturbed samples, so the feasibility rate should be read alongside the objective statistics.

## Mathematical model and local-search method

### Decision vector and cost components

Let the contract parameter vector be $\mathbf{x}=(x_1,\ldots,x_{15})$. For the four-tier example, $x_1,x_2,x_3$ are information-sharing coefficients, $x_4,x_5,x_6$ are cost-sharing ratios, and $x_7,\ldots,x_{15}$ are penalty-structure parameters. The code stores these groups consecutively in `params`.

The implementation-cost model is

$$
C(\mathbf{x}) = 100{,}000\sum_{i=1}^{3}x_i^2
+ 50{,}000\sum_{i=4}^{6}x_i
+ 30{,}000\sum_{i=7}^{15}x_i.
$$

The three terms represent information-system, cost-sharing administration, and penalty-structure implementation costs, respectively. In the code, `_calculate_implementation_cost` computes $C(\mathbf{x})$ in dollars per year. The model uses the stated coefficients as illustrative assumptions; it does not calculate an independent profit for each supply-chain participant.

### Amplification and objective

Let the baseline amplification be $B=3.2$ and let $\bar{x}=\frac{1}{3}\sum_{i=1}^{3}x_i$ be the mean information-sharing coefficient. The simplified amplification model is

$$
A(\mathbf{x})=\max\left\{B\left(1-0.6\bar{x}^{0.7}\right),\ 1.0\right\}.
$$

The exponent $0.7$ represents diminishing returns in this illustrative model, and the lower bound prevents modeled amplification from falling below $1.0$. The code implements this expression in `_calculate_bullwhip_amplification`.

The minimization objective is

$$
F(\mathbf{x}) = 0.7\frac{A(\mathbf{x})}{3.2}
+ 0.3\frac{C(\mathbf{x})}{1{,}000{,}000}.
$$

Both terms are dimensionless because amplification is divided by its baseline and cost is divided by one million dollars. The weights assign 0.7 to normalized amplification and 0.3 to normalized cost. Lower objective values are preferred. These weights express a modeling trade-off; they are not a claim that the two outcomes have identical units.

### Feasibility conditions

A candidate is feasible only if all of the following conditions hold:

$$
0\leq x_i\leq 1\quad (i=1,\ldots,15),\qquad
\frac{1}{3}\sum_{i=1}^{3}x_i\geq 0.1,
$$

$$
\sum_{i=4}^{6}x_i\leq 2.0,\qquad C(\mathbf{x})\leq 5{,}000{,}000.
$$

`_check_feasibility_constraints` checks these conditions. The objective assigns an infeasible vector an infinite value, so it cannot be accepted as an improvement over a feasible solution.

### Adaptive local search

At iteration $t$, the base perturbation magnitude is

$$
\delta_t=\max\left\{0.2(0.98)^t,\ 0.01\right\}.
$$

When more than 20 consecutive candidates have failed to improve the current solution, the magnitude used for the next candidate is increased to $\min\{1.5\delta_t,0.5\}$. The method selects $\lfloor 0.5p\rfloor$ distinct coordinates at random, where $p=15$, adds an independent uniform perturbation from the interval $[-\delta_t,\delta_t]$ to each selected coordinate, and clips the result to $[0,1]$.

If $\mathbf{x}^{(t)}$ is the current feasible solution and $\tilde{\mathbf{x}}$ is the perturbed candidate, the acceptance rule is

$$
\mathbf{x}^{(t+1)}=
\begin{cases}
\tilde{\mathbf{x}}, & F(\tilde{\mathbf{x}})<F(\mathbf{x}^{(t)}),\\
\mathbf{x}^{(t)}, & \text{otherwise}.
\end{cases}
$$

The search records the best solution found as well as the current accepted objective. Its base case is the feasible initialized solution; each iteration applies one perturbation and one objective evaluation. With $p$ parameters and $T$ iterations, the search performs $O(Tp)$ parameter-level work, apart from plotting and reporting. This describes computational growth, not a guarantee that the heuristic will find an optimum.

The implementation is divided among `_adaptive_perturbation`, `_objective_function`, and `optimize`. The convergence history includes the initial feasible objective followed by the current objective after each attempted iteration.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

# A fixed seed makes the demand generation and stochastic search reproducible.
np.random.seed(91)

print("Libraries imported successfully for iterative contract optimization.")

In [ ]:
class IterativeContractOptimizer:
    """Adaptive iterative local search for contract parameter optimization."""

    def __init__(self, n_tiers=4, n_months=12):
        if n_tiers != 4:
            raise ValueError("This example is defined for a four-tier supply chain.")
        if n_months <= 0:
            raise ValueError("n_months must be a positive integer.")

        self.n_tiers = n_tiers
        self.n_months = n_months

        # Demand is included as contextual data and is not used in the objective.
        self.base_demand = 1000
        self.demand_seasonality = 0.3
        self.demand_noise = 0.1
        self.demand_data = self._generate_demand_data()

        self.n_parameters = 15

        self.max_iterations = 500
        self.initial_perturbation = 0.2
        self.min_perturbation = 0.01
        self.perturbation_decay = 0.98
        self.no_improvement_limit = 50

        self.best_solution = None
        self.best_objective = float("inf")
        self.convergence_history = []

    def _generate_demand_data(self):
        """Generate monthly demand with annual seasonality and random noise."""
        demand = []
        for month in range(self.n_months):
            seasonal_factor = 1 + self.demand_seasonality * np.sin(
                2 * np.pi * month / 12
            )
            noise_factor = 1 + self.demand_noise * np.random.normal(0, 1)
            monthly_demand = self.base_demand * seasonal_factor * noise_factor
            demand.append(max(monthly_demand, 100))

        return np.array(demand)

    def _initialize_contract_parameters(self):
        """Draw a parameter vector from the example's initialization ranges."""
        params = np.zeros(self.n_parameters)

        # The first three entries control information sharing between tiers.
        for i in range(self.n_tiers - 1):
            params[i] = 0.5 + 0.2 * np.random.random()

        # The next three entries are upstream cost-sharing fractions.
        for i in range(self.n_tiers - 1):
            params[self.n_tiers - 1 + i] = 0.4 + 0.4 * np.random.random()

        # The remaining entries represent penalty-structure parameters.
        remaining_params = self.n_parameters - 2 * (self.n_tiers - 1)
        for i in range(remaining_params):
            params[2 * (self.n_tiers - 1) + i] = (
                0.1 + 0.3 * np.random.random()
            )

        return params

    def _calculate_bullwhip_amplification(self, params):
        """Evaluate the simplified amplification model for a parameter vector."""
        info_coeffs = params[: self.n_tiers - 1]
        base_amplification = 3.2
        info_effectiveness = np.mean(info_coeffs) ** 0.7
        contract_amplification = base_amplification * (
            1 - 0.6 * info_effectiveness
        )

        return max(contract_amplification, 1.0)

    def _calculate_implementation_cost(self, params):
        """Calculate annual implementation cost from the three parameter groups."""
        info_costs = np.sum(params[: self.n_tiers - 1] ** 2) * 100000
        sharing_costs = np.sum(
            params[self.n_tiers - 1 : 2 * (self.n_tiers - 1)]
        ) * 50000
        penalty_costs = np.sum(params[2 * (self.n_tiers - 1) :]) * 30000

        return info_costs + sharing_costs + penalty_costs

    def _check_feasibility_constraints(self, params):
        """Check parameter bounds and the model's aggregate feasibility limits."""
        if np.any(params < 0) or np.any(params > 1):
            return False

        info_coeffs = params[: self.n_tiers - 1]
        if np.mean(info_coeffs) < 0.1:
            return False

        cost_sharing = params[
            self.n_tiers - 1 : 2 * (self.n_tiers - 1)
        ]
        if np.sum(cost_sharing) > 2.0:
            return False

        if self._calculate_implementation_cost(params) > 5000000:
            return False

        return True

    def _objective_function(self, params):
        """Return the weighted objective, or infinity for an infeasible vector."""
        if not self._check_feasibility_constraints(params):
            return float("inf")

        amplification = self._calculate_bullwhip_amplification(params)
        implementation_cost = self._calculate_implementation_cost(params)
        normalized_amplification = amplification / 3.2
        normalized_cost = implementation_cost / 1000000

        return 0.7 * normalized_amplification + 0.3 * normalized_cost

    def _adaptive_perturbation(self, params, iteration, no_improvement_count):
        """Perturb a random subset of parameters with adaptive search strength."""
        base_perturbation = (
            self.initial_perturbation * self.perturbation_decay**iteration
        )
        current_perturbation = max(
            base_perturbation, self.min_perturbation
        )

        if no_improvement_count > 20:
            current_perturbation = min(current_perturbation * 1.5, 0.5)

        perturbed_params = params.copy()
        n_perturb = int(0.5 * len(params))
        perturb_indices = np.random.choice(
            len(params), n_perturb, replace=False
        )

        for idx in perturb_indices:
            perturbation = np.random.uniform(
                -current_perturbation, current_perturbation
            )
            perturbed_params[idx] += perturbation
            perturbed_params[idx] = np.clip(perturbed_params[idx], 0, 1)

        return perturbed_params

    def optimize(self):
        """Run adaptive local search and return the best feasible solution found."""
        print("Starting iterative contract parameter optimization...")

        # Rejection sampling keeps the starting point feasible without changing
        # the initialization ranges used for any individual parameter.
        current_params = self._initialize_contract_parameters()
        while not self._check_feasibility_constraints(current_params):
            current_params = self._initialize_contract_parameters()

        current_objective = self._objective_function(current_params)
        self.best_solution = current_params.copy()
        self.best_objective = current_objective
        self.convergence_history = [current_objective]

        no_improvement_count = 0
        start_time = time.time()

        for iteration in range(self.max_iterations):
            candidate_params = self._adaptive_perturbation(
                current_params, iteration, no_improvement_count
            )
            candidate_objective = self._objective_function(candidate_params)

            if candidate_objective < current_objective:
                current_params = candidate_params
                current_objective = candidate_objective
                no_improvement_count = 0

                if candidate_objective < self.best_objective:
                    self.best_solution = candidate_params.copy()
                    self.best_objective = candidate_objective
            else:
                no_improvement_count += 1

            self.convergence_history.append(current_objective)

            if no_improvement_count >= self.no_improvement_limit:
                print(
                    f"Stopped after {iteration + 1} iterations "
                    "without sufficient improvement."
                )
                break

            if (iteration + 1) % 50 == 0:
                print(
                    f"Iteration {iteration + 1}: "
                    f"Objective = {current_objective:.6f}"
                )

        optimization_time = time.time() - start_time
        print(f"\nOptimization completed in {optimization_time:.2f} seconds.")
        print(f"Best objective value: {self.best_objective:.6f}")

        return self.best_solution, self.best_objective

    def analyze_results(self, optimal_params):
        """Report modeled performance, parameter groups, and implementation cost."""
        print("\n=== OPTIMIZATION RESULTS ANALYSIS ===\n")

        info_coeffs = optimal_params[: self.n_tiers - 1]
        cost_sharing = optimal_params[
            self.n_tiers - 1 : 2 * (self.n_tiers - 1)
        ]
        penalty_structures = optimal_params[2 * (self.n_tiers - 1) :]

        final_amplification = self._calculate_bullwhip_amplification(
            optimal_params
        )
        implementation_cost = self._calculate_implementation_cost(
            optimal_params
        )
        baseline_amplification = 3.2
        reduction_percentage = (
            (baseline_amplification - final_amplification)
            / baseline_amplification
            * 100
        )

        print("BULLWHIP MITIGATION PERFORMANCE:")
        print(f"  Baseline amplification: {baseline_amplification:.2f}x")
        print(f"  Modeled amplification: {final_amplification:.2f}x")
        print(f"  Modeled reduction: {reduction_percentage:.1f}%\n")

        tier_names = [
            "Retailer to Manufacturer",
            "Manufacturer to Distributor",
            "Distributor to Supplier",
        ]
        print("INFORMATION-SHARING COEFFICIENTS:")
        for name, coefficient in zip(tier_names, info_coeffs):
            print(f"  {name}: {coefficient:.3f}")

        print("\nCOST-SHARING RATIOS (UPSTREAM / DOWNSTREAM):")
        for name, ratio in zip(tier_names, cost_sharing):
            upstream_share = ratio * 100
            downstream_share = (1 - ratio) * 100
            print(
                f"  {name}: {upstream_share:.1f}% upstream / "
                f"{downstream_share:.1f}% downstream"
            )

        print("\nPENALTY-STRUCTURE PARAMETERS:")
        for index, penalty in enumerate(penalty_structures, start=1):
            print(f"  Parameter {index}: {penalty:.3f}")

        print("\nIMPLEMENTATION COST:")
        print(f"  Total modeled annual cost: ${implementation_cost:,.0f}")
        print(
            f"  Average cost per tier: "
            f"${implementation_cost / self.n_tiers:,.0f}"
        )

        return {
            "amplification": final_amplification,
            "reduction_percentage": reduction_percentage,
            "impl_cost": implementation_cost,
            "info_coeffs": info_coeffs,
            "cost_sharing": cost_sharing,
            "penalty_structures": penalty_structures,
        }


print("IterativeContractOptimizer class defined successfully.")

In [ ]:
# Initialize the example and run the heuristic search.
optimizer = IterativeContractOptimizer(n_tiers=4, n_months=12)

print("=== ITERATIVE CONTRACT PARAMETER OPTIMIZATION ===")
print(f"Supply-chain tiers: {optimizer.n_tiers}")
print(f"Contract parameters: {optimizer.n_parameters}")
print(f"Maximum search iterations: {optimizer.max_iterations}")
print(f"Demand periods: {optimizer.n_months} months\n")

# These demand statistics describe the example input; demand is not used
# in the simplified objective or feasibility calculations.
print("DEMAND DATA CHARACTERISTICS:")
print(f"  Mean demand: {np.mean(optimizer.demand_data):.1f} units")
print(f"  Demand standard deviation: {np.std(optimizer.demand_data):.1f} units")
print(f"  Minimum demand: {np.min(optimizer.demand_data):.1f} units")
print(f"  Maximum demand: {np.max(optimizer.demand_data):.1f} units")
coefficient_of_variation = (
    np.std(optimizer.demand_data) / np.mean(optimizer.demand_data)
)
print(f"  Coefficient of variation: {coefficient_of_variation:.3f}\n")

optimal_params, optimal_objective = optimizer.optimize()

# These assertions connect the reported solution to the model definition.
assert optimizer._check_feasibility_constraints(optimal_params)
assert np.isclose(
    optimizer._objective_function(optimal_params), optimal_objective
)

## Interpret the selected contract

The next cell reports the best feasible vector found in this run. The information-sharing coefficients influence both modeled amplification and information-system cost. Cost-sharing ratios and penalty parameters influence the modeled implementation cost, but they do not change amplification in this formulation. Therefore, their reported values should be interpreted as outputs of the stated heuristic and cost model, not as proof of a participant's profit or as a unique economic contract.

The six-panel figure links the numerical results to the model: the convergence trace shows progress in the accepted objective; the parameter panels show the selected contract groups; the amplification comparison and cost breakdown expose the modeled trade-off; and the demand plot displays contextual data that is not used by the objective.

In [ ]:
results = optimizer.analyze_results(optimal_params)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    "Iterative Contract Parameter Optimization Results",
    fontsize=16,
    fontweight="bold",
)

# The initial value is at iteration zero; subsequent values follow each move.
ax1 = axes[0, 0]
iterations = np.arange(len(optimizer.convergence_history))
ax1.plot(
    iterations,
    optimizer.convergence_history,
    color="blue",
    linewidth=2,
    alpha=0.8,
)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Objective value (dimensionless)")
ax1.set_title("Accepted-Solution Objective")
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
tier_names = ["Retailer–Manufacturer", "Manufacturer–Distributor", "Distributor–Supplier"]
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1"]
bars = ax2.bar(tier_names, results["info_coeffs"], color=colors, alpha=0.8)
ax2.set_ylabel("Information-sharing coefficient")
ax2.set_title("Selected Information-Sharing Coefficients")
ax2.set_ylim(0, 1)
ax2.tick_params(axis="x", rotation=20)
ax2.grid(True, axis="y", alpha=0.3)
for bar, coefficient in zip(bars, results["info_coeffs"]):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{coefficient:.3f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax3 = axes[0, 2]
upstream_shares = results["cost_sharing"] * 100
downstream_shares = (1 - results["cost_sharing"]) * 100
x_positions = np.arange(len(tier_names))
width = 0.35
ax3.bar(
    x_positions - width / 2,
    upstream_shares,
    width,
    label="Upstream share",
    color="#FF6B6B",
    alpha=0.8,
)
ax3.bar(
    x_positions + width / 2,
    downstream_shares,
    width,
    label="Downstream share",
    color="#4ECDC4",
    alpha=0.8,
)
ax3.set_ylabel("Share of cost (%)")
ax3.set_title("Selected Cost-Sharing Ratios")
ax3.set_xticks(x_positions)
ax3.set_xticklabels(["R–M", "M–D", "D–S"])
ax3.legend()
ax3.grid(True, axis="y", alpha=0.3)

ax4 = axes[1, 0]
scenarios = ["Baseline", "Selected contract"]
amplifications = [3.2, results["amplification"]]
bars = ax4.bar(
    scenarios,
    amplifications,
    color=["#D9534F", "#5CB85C"],
    alpha=0.8,
)
ax4.set_ylabel("Bullwhip amplification factor")
ax4.set_title(
    "Modeled Amplification\n"
    f"Reduction: {results['reduction_percentage']:.1f}%"
)
ax4.grid(True, axis="y", alpha=0.3)
for bar, amplification in zip(bars, amplifications):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{amplification:.2f}x",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax5 = axes[1, 1]
info_cost = np.sum(optimal_params[:3] ** 2) * 100000
sharing_cost = np.sum(optimal_params[3:6]) * 50000
penalty_cost = np.sum(optimal_params[6:]) * 30000
costs = [info_cost, sharing_cost, penalty_cost]
cost_categories = [
    "Information systems",
    "Cost-sharing administration",
    "Penalty structures",
]
ax5.pie(
    costs,
    labels=cost_categories,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
)
ax5.set_title(
    "Modeled Annual Cost Breakdown\n"
    f"Total: ${results['impl_cost']:,.0f}"
)

ax6 = axes[1, 2]
months = np.arange(1, len(optimizer.demand_data) + 1)
ax6.plot(
    months,
    optimizer.demand_data,
    "o-",
    linewidth=2,
    markersize=6,
    color="purple",
    label="Monthly demand",
)
ax6.axhline(
    y=np.mean(optimizer.demand_data),
    color="red",
    linestyle="--",
    alpha=0.7,
    label="Mean demand",
)
ax6.set_xlabel("Month")
ax6.set_ylabel("Demand (units)")
ax6.set_title("Monthly Demand Context")
ax6.set_xticks(months)
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("The optimization results and model diagnostics are displayed above.")

## Sensitivity analysis

A heuristic solution can be sensitive to nearby parameter changes. This experiment adds independent Gaussian noise to every selected parameter, clips each value to $[0,1]$, and evaluates the perturbed vector with the same feasibility checks and objective. For each perturbation level, it reports the mean and standard deviation of the objective among feasible samples and the fraction of all samples that remain feasible.

This is a local robustness experiment, not a second optimization run. Infeasible samples are excluded from the objective statistics but remain in the denominator of the feasibility rate. If a perturbation level yields no feasible samples, its objective statistics are reported as unavailable.

In [ ]:
print("=== SENSITIVITY ANALYSIS ===")
print("Assessing nearby parameter vectors around the selected solution.\n")

perturbation_levels = [0.01, 0.05, 0.10, 0.15, 0.20]
n_samples = 50
robustness_results = []

for perturbation_level in perturbation_levels:
    sample_objectives = []

    for _ in range(n_samples):
        noise = np.random.normal(
            0, perturbation_level, len(optimal_params)
        )
        perturbed_params = np.clip(optimal_params + noise, 0, 1)
        objective = optimizer._objective_function(perturbed_params)

        if np.isfinite(objective):
            sample_objectives.append(objective)

    if sample_objectives:
        mean_objective = float(np.mean(sample_objectives))
        std_objective = float(np.std(sample_objectives))
    else:
        mean_objective = np.nan
        std_objective = np.nan

    robustness_results.append(
        {
            "perturbation": perturbation_level,
            "mean_objective": mean_objective,
            "std_objective": std_objective,
            "feasible_samples": len(sample_objectives),
        }
    )

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
perturbation_values = [
    result["perturbation"] for result in robustness_results
]
mean_objectives = [
    result["mean_objective"] for result in robustness_results
]
std_objectives = [
    result["std_objective"] for result in robustness_results
]

ax1.errorbar(
    perturbation_values,
    mean_objectives,
    yerr=std_objectives,
    marker="o",
    linewidth=2,
    markersize=8,
    capsize=5,
    label="Feasible-sample mean ± standard deviation",
)
ax1.axhline(
    y=optimal_objective,
    color="red",
    linestyle="--",
    alpha=0.7,
    label="Selected-solution objective",
)
ax1.set_xlabel("Gaussian perturbation standard deviation")
ax1.set_ylabel("Objective value (dimensionless)")
ax1.set_title("Objective Sensitivity")
ax1.legend()
ax1.grid(True, alpha=0.3)

feasible_rates = [
    result["feasible_samples"] / n_samples * 100
    for result in robustness_results
]
ax2.bar(perturbation_values, feasible_rates, color="green", alpha=0.7)
ax2.set_xlabel("Gaussian perturbation standard deviation")
ax2.set_ylabel("Feasible samples (%)")
ax2.set_title("Feasibility Under Perturbation")
ax2.set_ylim(0, 110)
ax2.grid(True, axis="y", alpha=0.3)
for perturbation, rate in zip(perturbation_values, feasible_rates):
    ax2.text(
        perturbation,
        rate + 2,
        f"{rate:.1f}%",
        ha="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

print("Perturbation | Mean objective | Std. dev. | Feasible samples")
print("-" * 66)
for result in robustness_results:
    if result["feasible_samples"]:
        objective_summary = (
            f"{result['mean_objective']:.6f} | "
            f"{result['std_objective']:.6f}"
        )
    else:
        objective_summary = "unavailable | unavailable"

    print(
        f"{result['perturbation']:11.3f} | {objective_summary} | "
        f"{result['feasible_samples']:3d}/{n_samples}"
    )

smallest_perturbation_result = robustness_results[0]
if (
    smallest_perturbation_result["feasible_samples"] > 0
    and optimal_objective != 0
):
    relative_std = (
        smallest_perturbation_result["std_objective"]
        / abs(optimal_objective)
        * 100
    )
    print(
        "\nAt the smallest perturbation level, the feasible-sample "
        f"objective standard deviation is {relative_std:.2f}% of the "
        "selected-solution objective."
    )
else:
    print(
        "\nRelative objective variation is unavailable because the "
        "smallest perturbation level has no feasible samples or the "
        "selected objective is zero."
    )

## Computational-efficiency analysis

The convergence history contains the initial objective at index zero and one entry per attempted move. Consequently, the number of attempted iterations is `len(convergence_history) - 1`. The milestones below measure when the search first achieves a stated fraction of the total improvement observed between the initial and final current objectives. They describe this run; they do not establish a general convergence rate or guarantee.

In [ ]:
print("=== COMPUTATIONAL EFFICIENCY ANALYSIS ===\n")

convergence_iterations = len(optimizer.convergence_history) - 1
initial_objective = optimizer.convergence_history[0]
final_objective = optimizer.convergence_history[-1]
total_improvement = initial_objective - final_objective

print(f"Attempted iterations: {convergence_iterations}")
print(f"Initial objective: {initial_objective:.6f}")
print(f"Final current objective: {final_objective:.6f}")

if initial_objective != 0:
    improvement_percentage = total_improvement / abs(initial_objective) * 100
else:
    improvement_percentage = np.nan

if np.isfinite(improvement_percentage):
    print(f"Observed objective improvement: {improvement_percentage:.2f}%")
else:
    print("Observed percentage improvement is unavailable for a zero initial objective.")

improvement_milestones = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
milestone_iterations = []

if total_improvement > 0:
    for milestone in improvement_milestones:
        target_objective = initial_objective - milestone * total_improvement
        for index, objective in enumerate(optimizer.convergence_history):
            if objective <= target_objective:
                milestone_iterations.append(index)
                break
        else:
            milestone_iterations.append(convergence_iterations)

    print("\nImprovement milestone | Iterations reached | Share of run")
    print("-" * 61)
    for milestone, iterations_needed in zip(
        improvement_milestones, milestone_iterations
    ):
        if convergence_iterations > 0:
            share_of_run = iterations_needed / convergence_iterations * 100
        else:
            share_of_run = 0.0
        print(
            f"{milestone * 100:19.0f}% | {iterations_needed:18d} | "
            f"{share_of_run:10.1f}%"
        )

    if convergence_iterations > 0:
        print(
            "\nIterations per percentage point of observed improvement: "
            f"{convergence_iterations / improvement_percentage:.2f}"
        )
        print(
            "Observed percentage improvement per iteration: "
            f"{improvement_percentage / convergence_iterations:.3f}%"
        )
        reached_90_percent_early = (
            milestone_iterations[3] <= convergence_iterations / 2
        )
        print(
            "Reached 90% of observed improvement in the first half "
            f"of the run: {'Yes' if reached_90_percent_early else 'No'}"
        )
    else:
        print("No search iterations were attempted.")
else:
    print(
        "\nNo positive improvement occurred between the initial and final "
        "current objectives, so improvement milestones are not applicable."
    )

## Method interpretation and limitations

### What this heuristic is useful for

Adaptive local search provides a practical way to explore a bounded parameter vector when a candidate can be evaluated by a known objective and feasibility test. This example makes the search mechanics visible: a feasible starting vector is perturbed, candidates are evaluated, improving candidates are accepted, and perturbation strength responds to iteration progress and stagnation.

### What the results do and do not establish

- The reported solution is the best feasible vector encountered by this seeded run, not a proven global optimum.
- The no-improvement stopping rule is a computational stopping criterion, not a mathematical convergence guarantee.
- The modeled amplification is a function of the information-sharing coefficients alone. The generated demand data does not calibrate that function or enter the objective.
- The feasibility checks enforce parameter and aggregate cost limits. They do not calculate or guarantee positive profits for individual participants.
- Cost-sharing ratios and penalty parameters affect the cost expression but have no direct benefit term in the amplification model. Their selected values therefore depend on the defined cost terms, constraints, initialization, and stochastic search.
- One seeded run illustrates reproducible execution in a fixed environment. Different seeds or search settings can lead to different heuristic solutions.

Natural extensions within the same modeling framework include adding participant-specific profit constraints, calibrating the amplification relationship with supply-chain data, or testing larger parameter spaces. Those extensions require explicit new data and model assumptions; they are not part of the present calculation.